In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
import cftime
import sys
import os
import pandas as pd

calc_annual = False
read_last_n = False
read_first_n = False
nfiles = 170

#Additional plots
plot_total = True

# Conversion factors
km2_to_m2 = 1e6
kg_to_Gt = 1e-12
CO2_to_C = 12.011 / 44.01  # Conversion factor from CO2 to C
seconds_per_year = 365 * 24 * 60 * 60
days_per_year = 365
days_per_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
sec_per_month = [days * 24 * 60 * 60 for days in days_per_month]


variables = ["FCO2"]

case_dir = '/datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist'

# Find all timeseries files
timeseries_files = sorted(glob.glob(f'{case_dir}/*.clm2.h0a*-*.nc'))

# Initialize a dictionary to store the accumulated results
results = {var: [] for var in variables}
time = []
co2_conc = []
fates_LU_area = []
if calc_annual:
    timeseries_files = timeseries_files[:-1]
if read_last_n:
    timeseries_files = timeseries_files[-nfiles:]
elif read_first_n:
    timeseries_files = timeseries_files[:nfiles]



In [2]:
# Loop through all timeseries files
first_file=True
for filename in timeseries_files[:]:
    print(f"Processing file: {filename}")
    mon=filename.split('.')[-2][-2:]
    mon_index=int(mon)-1
    print(f"Month index: {mon_index}")
    try:
        with xr.open_dataset(filename, engine='netcdf4', decode_timedelta=True) as data:            
            if first_file:
                area = data["area"].fillna(0)
                landfrac = data["landfrac"].fillna(0)               
                
            for var in variables:
                if var in data:
                    var_data_in = data[var]                                        
                    var_data = var_data_in
                    
                    # Determine spatial dimensions
                    
                    spatial_dims = ("lndgrid",)
                    
                    if first_file:
                        print(f'Reading {var}, with units: {data[var].units}')
                    # Carbon pools: multiply by area and landfrac (convert m^2 to km^2)
                    area_land_m2 = area * km2_to_m2 * landfrac
                    
                    total = (var_data * area_land_m2 * kg_to_Gt * CO2_to_C * sec_per_month[mon_index]).sum(dim=spatial_dims)
                    results[var].append(total.values)
                    
            first_file = False
            time.append(data['time'].values)
    except FileNotFoundError:
        print(f"File not found: {filename}")
    except ValueError as e:
        print(f"Error reading the file: {e}")



Processing file: /datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist/noresm-fates-ne16-LU-PPE-1850-1900.2026-05-02.clm2.h0a.1851-01.nc
Month index: 0
Reading FCO2, with units: kgCO2/m2/s
Processing file: /datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist/noresm-fates-ne16-LU-PPE-1850-1900.2026-05-02.clm2.h0a.1851-02.nc
Month index: 1
Processing file: /datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist/noresm-fates-ne16-LU-PPE-1850-1900.2026-05-02.clm2.h0a.1851-03.nc
Month index: 2
Processing file: /datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist/noresm-fates-ne16-LU-PPE-1850-1900.2026-05-02.clm2.h0a.1851-04.nc
Month index: 3
Processing file: /datalake/NS9188K/share/jessien/may26/noresm-fates-ne16-LU-PPE-1901-2024.2026-05-03/lnd/hist/noresm-fates-ne16-LU-PPE-1850-1900.2026-05-02.clm2.h0a.1851-05.nc
Month index: 4
Processi

In [3]:
# Convert lists to numpy arrays
for var in variables:
    print(f"Processing {var}")
    if len(results[var]) > 0:
        results[var] = np.concatenate(results[var])
    else:
        results[var] = np.array([])

if len(time) > 0:
    time = np.concatenate(time)
else:
    time = np.array([])

if calc_annual:
    annual_means = {var: [] for var in variables}
    annual_time = []
    for year in np.unique(time.astype('datetime64[Y]').astype(int)):
        mask = (time.astype('datetime64[Y]').astype(int) == year)
        annual_time.append(time[mask][0])
        for var in variables:
            annual_means[var].append(results[var][mask].mean())
    time = np.array(annual_time)
    for var in variables:
        results[var] = np.array(annual_means[var])

# Calculate deltas
deltas = {var: np.diff(results[var], prepend=np.nan) for var in variables}



Processing FCO2


In [4]:
# save the results as a csv file
df = pd.DataFrame({
    'time': time,
    'FCO2': results['FCO2'],
    'delta_FCO2': deltas['FCO2']
})
df.to_csv('nbp_timeseries_mayrun.csv', index=False)